# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muhammad-Ahmed-Zia/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I'm training two models, not one, specifically to satisfy "does not
reward complexity alone": Logistic Regression (simple, fast, fully
interpretable coefficients) and Random Forest (handles non-linear
interactions between signals, matches what already beat the hand rule
3x on the starter dataset back in ML-01). If Random Forest doesn't
meaningfully beat Logistic Regression, that's a real finding, not a
failure — it means the extra complexity isn't earning its place, and
I'd report Logistic Regression as the honest choice instead.

Label: is_declining (avg_impr_second_half < avg_impr_first_half), the
same proxy used since ML-04 — the closest thing I have to ground truth
for "this page needed action," since neither my Week-4 baseline nor
this week's model has access to a true future outcome.

Features used (all current-window, non-label-derived):
avg_daily_impressions, avg_position, days_with_impressions,
content_age_days, avg_impr_first_half, ctr, ctr_gap, position_tier
(one-hot). avg_impr_second_half is explicitly EXCLUDED — it's what
the label is built from, including it would be the same leakage
lesson from Notebook 02.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [20]:
# Rebuild features using ONLY first-half data — nothing from the
# second half (the label's target window) is allowed into any feature.
base = con.sql(f"""
    SELECT
        content_hash_id, client_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date < DATE '{MONTH}-16') AS impressions,
        SUM(gsc_clicks) FILTER (WHERE report_date < DATE '{MONTH}-16') AS clicks,
        AVG(gsc_avg_position) FILTER (WHERE report_date < DATE '{MONTH}-16') AS avg_position,
        COUNT(*) FILTER (WHERE gsc_impressions > 0 AND report_date < DATE '{MONTH}-16') AS days_with_impressions,
        AVG(gsc_impressions) FILTER (WHERE report_date < DATE '{MONTH}-16') AS avg_impr_first_half,
        AVG(gsc_impressions) FILTER (WHERE report_date >= DATE '{MONTH}-16') AS avg_impr_second_half
    FROM read_parquet('{REL}/**/fact_content_daily_performance/**/month={MONTH}/*.parquet')
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) FILTER (WHERE report_date < DATE '{MONTH}-16') > 0
""").df()
base["ctr"] = base["clicks"] / base["impressions"]

def tier(p):
    if p <= 3: return "top_3"
    elif p <= 10: return "page_1"
    elif p <= 20: return "striking"
    elif p <= 50: return "page_3_5"
    else: return "deep"
base["position_tier"] = base["avg_position"].apply(tier)

tier_avg_ctr = base.groupby("position_tier")["ctr"].transform("mean")
base["ctr_gap"] = (tier_avg_ctr - base["ctr"]).clip(lower=0)
base["baseline_score"] = base["impressions"] * base["ctr_gap"]  # now first-half-only, fair vs model

base = base.dropna(subset=["avg_impr_first_half", "avg_impr_second_half"])
base["is_declining"] = (base["avg_impr_second_half"] < base["avg_impr_first_half"]).astype(int)

dim = con.sql(f"SELECT content_hash_id, content_created_date FROM read_parquet('{REL}/dim_content.parquet')").df()
base = base.merge(dim, on="content_hash_id", how="left")
base["content_age_days"] = (pd.Timestamp(f"{MONTH}-01") - pd.to_datetime(base["content_created_date"])).dt.days

print("Rows after cleaning:", len(base))
print("Declining rate:", base["is_declining"].mean().round(3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows after cleaning: 151980
Declining rate: 0.498


In [21]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(base, groups=base["client_hash_id"]))
df_train, df_test = base.iloc[train_idx].copy(), base.iloc[test_idx].copy()

print(f"Train: {len(df_train)} rows, {df_train['client_hash_id'].nunique()} clients")
print(f"Test:  {len(df_test)} rows, {df_test['client_hash_id'].nunique()} clients")

Train: 137460 rows, 33 clients
Test:  14520 rows, 11 clients


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

feature_cols = ["avg_daily_impressions" if "avg_daily_impressions" in base.columns else "impressions",
                "avg_position", "days_with_impressions", "content_age_days",
                "avg_impr_first_half", "ctr", "ctr_gap"]
feature_cols = [c for c in feature_cols if c in base.columns]

X_train = pd.get_dummies(df_train[feature_cols + ["position_tier"]], columns=["position_tier"]).fillna(0)
X_test = pd.get_dummies(df_test[feature_cols + ["position_tier"]], columns=["position_tier"]).fillna(0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)  # align dummy columns
y_train, y_test = df_train["is_declining"], df_test["is_declining"]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(max_iter=2000).fit(X_train_scaled, y_train)
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1).fit(X_train, y_train)

lr_scores = lr.predict_proba(X_test_scaled)[:, 1]
rf_scores = rf.predict_proba(X_test)[:, 1]
baseline_scores = df_test["baseline_score"].values  # SAME test rows as the models

results = {}
for name, scores in [("Baseline (ML-07 CTR-gap rule)", baseline_scores),
                      ("Logistic Regression", lr_scores),
                      ("Random Forest", rf_scores)]:
    p50 = precision_at_k(scores, y_test.values, 50)
    ap = average_precision_score(y_test, scores)
    results[name] = {"Precision@50": round(p50, 3), "Average Precision": round(ap, 3)}

comparison = pd.DataFrame(results).T
print(comparison)

                               Precision@50  Average Precision
Baseline (ML-07 CTR-gap rule)          0.56              0.561
Logistic Regression                    0.58              0.553
Random Forest                          0.66              0.601


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [23]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, scoring="average_precision")
importance_df = pd.DataFrame({
    "feature": X_test.columns, "importance": perm.importances_mean
}).sort_values("importance", ascending=False)
print(importance_df)

# Look at where the winning model disagrees most with the baseline
df_test = df_test.reset_index(drop=True)
df_test["rf_score"] = rf_scores
df_test["baseline_rank"] = df_test["baseline_score"].rank(ascending=False)
df_test["rf_rank"] = df_test["rf_score"].rank(ascending=False)
df_test["rank_gap"] = (df_test["baseline_rank"] - df_test["rf_rank"]).abs()

biggest_disagreements = df_test.sort_values("rank_gap", ascending=False).head(10)
print(biggest_disagreements[["content_hash_id", "position_tier", "is_declining",
                               "baseline_rank", "rf_rank", "ctr_gap"]])

                   feature  importance
3         content_age_days    0.052665
2    days_with_impressions    0.029968
5                      ctr    0.010865
4      avg_impr_first_half    0.007962
0              impressions    0.006942
1             avg_position    0.006098
6                  ctr_gap    0.003399
7       position_tier_deep    0.000131
10  position_tier_striking    0.000085
11     position_tier_top_3   -0.000155
8     position_tier_page_1   -0.000447
9   position_tier_page_3_5   -0.000719
                content_hash_id position_tier  is_declining  baseline_rank  \
6363   content_322123507238fc2a         top_3             0          262.0   
6423   content_f8ce8bf4fdd0aacf         top_3             0          421.5   
7143   content_9adc0515f370d5e2         top_3             0          716.0   
4631   content_58d6242b12aa1bb2         top_3             0          625.0   
4660   content_1ac154df478ca038         top_3             0          391.0   
4653   content_00fa47e68d

Random Forest beat the baseline at Precision@50 by 1.18x (0.66 vs
0.56) and modestly on Average Precision (0.601 vs 0.561). Logistic
Regression, once properly scaled, actually matched the baseline
closely (0.58 vs 0.56 Precision@50, 0.553 vs 0.561 Average
Precision) — a real, honest result: LR is roughly tied with the hand
rule, not clearly better or worse.

The complexity question has a real, honest answer: Random Forest's
extra complexity DID earn a modest edge over both the baseline and
Logistic Regression. But that edge is 1.18x, not the 3x lift seen on
the starter dataset back in ML-01 — real warehouse data with this
proxy label is a harder problem than the clean starter slice, and
the honest gain here is real but modest, not dramatic.

Permutation importance holds a genuine surprise: content_age_days is
the top feature (0.053), well ahead of ctr_gap (0.003) — the exact
signal the baseline is built entirely around. This means Random
Forest isn't winning by using the baseline's own signal more
cleverly; it's leaning on a mostly different one. That's either a
real, useful finding the baseline is missing, or a sign the model
found a spurious correlation with page age that happens to predict
this month's proxy label — I'd treat "content age matters" as
directional, not proven, without testing it on a second month.

Looking at the biggest baseline-vs-model disagreements: all ten are
pages the baseline ranked as moderate-priority (rank ~262-777 of
~14,520) based on a real CTR gap, that were NOT actually declining
(is_declining=0) — and Random Forest correctly pushed every one of
them to the very bottom of its ranking (rank ~13,970-14,370) instead.
This suggests part of Random Forest's advantage isn't finding new
true positives the rule missed, but avoiding a specific class of
false positive: pages with a genuine CTR shortfall that simply
aren't declining for other reasons the rule can't see.

This does NOT prove the model is right and the baseline is wrong —
both are scored against the same imperfect is_declining proxy, not a
true future outcome. What it shows is where the two methods would
send a reviewer to different pages first, and that Random Forest
appears to filter out a specific baseline blind spot rather than
uniformly outperforming it everywhere.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.